In [ ]:
import polars as pl
from tqdm import tqdm

In [ ]:
anno_df = pl.read_parquet('/s/project/deeprvat/wgs_preprocessing_rap/annotation/wgs_78_genes_annotations.parquet')
anno_df

In [ ]:
anno_df[['chrom', 'pos', 'ref', 'alt', 'id', 'col', 'Gene', 'region', 'AF_ukb', ]]

In [ ]:
s = pl.read_parquet('/s/project/absplice/AbSplice_2_0/data/results/snv_avsplice2_all_max_preds_NEW/hg38/model_27/ENSG00000169174_max_preds.parquet')
s

In [ ]:

absp_dir = '/s/project/absplice/AbSplice_2_0/data/results/snv_avsplice2_all_max_preds_NEW/hg38/model_27'

vm = anno_df[['chrom', 'pos', 'ref', 'alt', 'id', 'col', 'Gene', 'region', 'AF_ukb', ]].lazy()

absp_list = []
skip_list = []
for gene_id in tqdm(anno_df['region'].unique()):
    try:

        scores_gene = pl.scan_parquet(f'{absp_dir}/{gene_id}_max_preds.parquet').with_columns(
            pl.col('chrom').cast(pl.Utf8),
            pl.col('ref').cast(pl.Utf8),
            pl.col('alt').cast(pl.Utf8),
            pl.col('gene_id').cast(pl.Utf8).alias("region"),
        ).rename({'end':'pos'}).drop(['gene_id', 'start'])

        # sampled_gene = vm.filter(pl.col('region') == gene_id).lazy()

        scores_lazy_gene = vm.filter(pl.col('region') == gene_id).join(
            scores_gene, 
            on = ['chrom','pos','ref','alt', 'region'],
            how = 'left',
            )

        absp_list.append(scores_lazy_gene.collect())
        # break
    
    except FileNotFoundError:
        print(f"File not for gene: {gene_id}")
        skip_list.append(gene_id)

all_scores_df = pl.concat(absp_list)
all_scores_df

In [ ]:
absp2_scores_df = vm.collect().join(all_scores_df, on = ['chrom','pos','ref','alt', 'id', 'col', 'Gene', 'region', 'AF_ukb'], how = 'left')
absp2_scores_df

In [ ]:
absp2_scores_df['region'].n_unique()

In [ ]:
absp2_scores_df.lazy().sink_parquet('/s/project/deeprvat/wgs_preprocessing_rap/annotation/wgs_78_genes_absplice2.parquet')

In [ ]:
anno_df.filter(pl.col('region').is_in(skip_list))

In [ ]:
anno_df.shape[0] - 191_134